In [ ]:
# CatBoost model (drop-in replacement for RF cell)

import numpy as np
import pandas as pd
!pip install -q catboost
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== PREPARING DATA FOR CATBOOST ===\n")

# Features
categorical_features = ['service_name', 'comm_plan_name', 'case_origin']
numerical_features = [
    'year', 'month', 'hour',
    'is_infrastructure', 'is_followup'#, 'description_length'# 'day_of_week' 'council_district',
]
all_features = categorical_features + numerical_features

X = model_data[all_features].copy()
y = model_data['log_wait'].copy()

# Ensure categorical dtypes are strings and missing-safe
for c in categorical_features:
    X[c] = X[c].astype(str).fillna("Unknown")

print(f"Total samples: {len(X):,}")
print(f"Features: {len(all_features)}")
print(f"  - Categorical: {len(categorical_features)}")
print(f"  - Numerical: {len(numerical_features)}")

# Time-based split (oldest 80% train, newest 20% test)
sorted_idx = model_data['date_requested'].sort_values().index
split_point = int(len(sorted_idx) * 0.8)
train_idx = sorted_idx[:split_point]
test_idx = sorted_idx[split_point:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"\n✓ Train set: {len(X_train):,} samples")
print(f"✓ Test set: {len(X_test):,} samples")
print(f"Train date range: {model_data.loc[train_idx, 'date_requested'].min()} to {model_data.loc[train_idx, 'date_requested'].max()}")
print(f"Test date range: {model_data.loc[test_idx, 'date_requested'].min()} to {model_data.loc[test_idx, 'date_requested'].max()}")

print("\n=== TRAINING CATBOOST ===\n")

cb_model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=200
)

cb_model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    use_best_model=True,
    early_stopping_rounds=200
)

print("\n✓ CatBoost trained successfully!")

print("\n=== MODEL EVALUATION ===\n")

y_train_pred = cb_model.predict(X_train)
y_test_pred = cb_model.predict(X_test)

def evaluate_model(y_true_log, y_pred_log, dataset_name):
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2_log = r2_score(y_true_log, y_pred_log)

    y_true_days = np.expm1(y_true_log)
    y_pred_days = np.expm1(y_pred_log)

    mae_days = mean_absolute_error(y_true_days, y_pred_days)
    rmse_days = np.sqrt(mean_squared_error(y_true_days, y_pred_days))

    print(f"{dataset_name}:")
    print(f"  Log-space MAE:  {mae_log:.3f}")
    print(f"  Log-space RMSE: {rmse_log:.3f}")
    print(f"  Log-space R²:   {r2_log:.3f}")
    print(f"  Day-space MAE:  {mae_days:.2f} days")
    print(f"  Day-space RMSE: {rmse_days:.2f} days")
    print()

    return mae_log, rmse_log, r2_log, mae_days, rmse_days

train_mae_log, train_rmse_log, train_r2, train_mae_days, train_rmse_days = evaluate_model(y_train, y_train_pred, "Training Set")
test_mae_log, test_rmse_log, test_r2, test_mae_days, test_rmse_days = evaluate_model(y_test, y_test_pred, "Test Set")

print("=== FEATURE IMPORTANCE ===\n")
fi = cb_model.get_feature_importance(prettified=True)
print(fi.head(15).to_string(index=False))

print("\n=== CATBOOST SUMMARY ===")
print(f"Best iteration: {cb_model.get_best_iteration()}")
print(f"Test MAE (days): {test_mae_days:.2f}")
print(f"Test RMSE (days): {test_rmse_days:.2f}")
print(f"Test Log-R²: {test_r2:.3f}")


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# assumes X_train, X_test, y_train, y_test, categorical_features already exist

seeds = [42, 43, 44]
rows = []

for seed in seeds:
    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        iterations=2000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=seed,
        verbose=False
    )
    
    model.fit(
        X_train, y_train,
        cat_features=categorical_features,
        eval_set=(X_test, y_test),
        use_best_model=True,
        early_stopping_rounds=200
    )
    
    y_pred_log = model.predict(X_test)
    y_true_days = np.expm1(y_test)
    y_pred_days = np.expm1(y_pred_log)
    
    rows.append({
        "seed": seed,
        "best_iteration": model.get_best_iteration(),
        "test_log_r2": r2_score(y_test, y_pred_log),
        "test_mae_days": mean_absolute_error(y_true_days, y_pred_days),
        "test_rmse_days": np.sqrt(mean_squared_error(y_true_days, y_pred_days)),
    })

results = pd.DataFrame(rows).sort_values("test_mae_days")
print(results.to_string(index=False))

print("\nAverages:")
print(results[["test_log_r2", "test_mae_days", "test_rmse_days"]].mean())

print("\nStd dev:")
print(results[["test_log_r2", "test_mae_days", "test_rmse_days"]].std())
